# 102 — Stochastic Bootstrap Ensemble with Uncertainty

**Motivation:** A single LGBM trained on all data gives a point prediction. But different bootstrap samples of the training data produce different models. The **mean** of 50 bootstrapped LGBMs is a better predictor (reduced variance), and the **std** is an uncertainty estimate.

**Strategy:**
1. Train 50 LGBM models, each on a bootstrap sample of training data (63.2% unique compounds on average)
2. Predict on: (a) full training set (OOF via out-of-bag samples), (b) test set
3. Use mean as the ensemble prediction, std as uncertainty
4. Also try weighting predictions by 1/uncertainty in the grand ensemble context
5. Save OOF/test arrays + uncertainty arrays for downstream use

**OOF construction:** For each compound, average predictions only from models where that compound was NOT in the bootstrap sample (true out-of-bag). This gives unbiased OOF estimates.

**Expected gain:** ~3-5% RAE reduction vs single LGBM. Uncertainty quantification is a bonus for ensemble weighting.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} "
              f"r={pr:.4f} rho={sp:.4f} tau={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 0


In [4]:
# ---- Bootstrap ensemble: train 50 LGBM models ----
N_BOOTSTRAP = 50
N_TRAIN = len(y_tr)
N_TEST  = len(te)

# Arrays to accumulate predictions
oob_preds   = np.zeros(N_TRAIN, dtype=np.float64)  # OOB sum
oob_counts  = np.zeros(N_TRAIN, dtype=np.int32)    # OOB count per sample
oob_sq      = np.zeros(N_TRAIN, dtype=np.float64)  # for variance: sum of squares
te_preds_all = np.zeros((N_BOOTSTRAP, N_TEST), dtype=np.float64)

# Faster LGBM hyperparams for bootstrap (no early stopping needed,
# each model sees ~63% of data so use fewer trees)
BOOT_LGBM = dict(n_estimators=400, num_leaves=63, learning_rate=0.05,
                 min_child_samples=8, subsample=0.85, colsample_bytree=0.8,
                 reg_alpha=0.1, reg_lambda=0.1, verbose=-1, n_jobs=4)

print(f"Training {N_BOOTSTRAP} bootstrap LGBM models...", flush=True)
rng_boot = np.random.default_rng(SEED)

for b in range(N_BOOTSTRAP):
    boot_idx = rng_boot.integers(0, N_TRAIN, size=N_TRAIN)  # bootstrap sample
    oob_mask = np.ones(N_TRAIN, dtype=bool)
    oob_mask[np.unique(boot_idx)] = False  # True = OOB (not seen in this bootstrap)
    oob_idx = np.where(oob_mask)[0]

    seed_b = int(SEED + b * 7)
    m = lgb.LGBMRegressor(**{**BOOT_LGBM, "random_state": seed_b})
    m.fit(X_tr[boot_idx], y_tr[boot_idx], callbacks=[lgb.log_evaluation(-1)])

    # OOB predictions
    if len(oob_idx) > 0:
        p_oob = m.predict(X_tr[oob_idx])
        oob_preds[oob_idx]  += p_oob
        oob_sq[oob_idx]     += p_oob ** 2
        oob_counts[oob_idx] += 1

    # Test predictions
    te_preds_all[b] = m.predict(X_te)

    if (b+1) % 10 == 0:
        # Compute interim OOF RAE on samples with >= 5 OOB votes
        enough = oob_counts >= 5
        oof_interim = np.where(oob_counts>0, oob_preds/oob_counts, np.nan)
        r_interim = rae(y_tr[enough], oof_interim[enough])
        print(f"  [{b+1:2d}/{N_BOOTSTRAP}] interim OOF RAE (n_oob>={5})={r_interim:.4f}  "
              f"avg_oob_count={oob_counts.mean():.1f}", flush=True)

print("Bootstrap training complete.", flush=True)


Training 50 bootstrap LGBM models...


  [10/50] interim OOF RAE (n_oob>=5)=0.5434  avg_oob_count=3.7


  [20/50] interim OOF RAE (n_oob>=5)=0.5512  avg_oob_count=7.4


  [30/50] interim OOF RAE (n_oob>=5)=0.5487  avg_oob_count=11.1


  [40/50] interim OOF RAE (n_oob>=5)=0.5485  avg_oob_count=14.7


  [50/50] interim OOF RAE (n_oob>=5)=0.5475  avg_oob_count=18.4


Bootstrap training complete.


In [5]:
# ---- OOB prediction aggregation ----
print("\n=== OOB Prediction Analysis ===", flush=True)

# Mean OOB prediction per compound
oof_oob_mean = np.where(oob_counts > 0, oob_preds / oob_counts, np.nan)
# OOB variance (Var = E[X^2] - (E[X])^2)
oof_oob_var  = np.where(oob_counts > 1,
                        oob_sq/oob_counts - (oob_preds/oob_counts)**2,
                        np.nan)
oof_oob_std  = np.sqrt(np.maximum(oof_oob_var, 0))

print(f"OOB coverage: {(oob_counts>0).sum():,}/{N_TRAIN} compounds have >= 1 OOB vote")
print(f"OOB votes: mean={oob_counts.mean():.1f}  min={oob_counts.min()}  max={oob_counts.max()}")
print(f"OOB uncertainty (std): mean={np.nanmean(oof_oob_std):.4f}  "
      f"median={np.nanmedian(oof_oob_std):.4f}")

# For compounds with 0 OOB votes, fall back to full-ensemble test prediction
# (shouldn't happen for N_BOOTSTRAP=50 but let's be safe)
if (oob_counts == 0).any():
    n_zero = (oob_counts==0).sum()
    print(f"WARNING: {n_zero} compounds have 0 OOB votes, using train mean fallback")
    oof_oob_mean[oob_counts == 0] = y_tr.mean()
    oof_oob_std[oob_counts == 0]  = float(y_tr.std())

m_oob = full_metrics(y_tr[oob_counts>0], oof_oob_mean[oob_counts>0],
                     label="oob_mean (n_oob>0)")



=== OOB Prediction Analysis ===


OOB coverage: 4,139/4139 compounds have >= 1 OOB vote
OOB votes: mean=18.4  min=7  max=30
OOB uncertainty (std): mean=0.1792  median=0.1676
  [oob_mean (n_oob>0)] RAE=0.5475 MAE=0.4981 R2=0.6189 r=0.7879 rho=0.7476 tau=0.5547


In [6]:
# ---- Scaffold 5-fold CV for calibrated OOF (augment OOB with fold-aware results) ----
# OOB is already a good OOF estimate, but scaffold-split validation is our standard.
# Here we train 5 fold models with full bootstrap-ensemble approach.
print("\n=== Scaffold 5-fold CV (bootstrap within each fold) ===", flush=True)
N_BOOT_FOLD = 20  # smaller N per fold for speed
oof_scaffold = np.full(N_TRAIN, np.nan)
oof_scaffold_std = np.full(N_TRAIN, np.nan)

for fold, (tr_idx, va_idx) in enumerate(splits):
    fold_preds = np.zeros((N_BOOT_FOLD, len(va_idx)), dtype=np.float64)
    rng_f = np.random.default_rng(SEED + fold * 13)

    for b in range(N_BOOT_FOLD):
        bidx = tr_idx[rng_f.integers(0, len(tr_idx), size=len(tr_idx))]
        seed_b = int(SEED + fold*100 + b)
        m = lgb.LGBMRegressor(**{**BOOT_LGBM, "random_state": seed_b})
        m.fit(X_tr[bidx], y_tr[bidx], callbacks=[lgb.log_evaluation(-1)])
        fold_preds[b] = m.predict(X_tr[va_idx])

    oof_scaffold[va_idx]     = fold_preds.mean(0)
    oof_scaffold_std[va_idx] = fold_preds.std(0)
    r = rae(y_tr[va_idx], oof_scaffold[va_idx])
    print(f"  fold {fold+1}  RAE={r:.4f}  uncertainty_mean={fold_preds.std(0).mean():.4f}",
          flush=True)

m_scaffold = full_metrics(y_tr, oof_scaffold, cliff_pairs, "scaffold_bootstrap")
m_scaffold_a = full_metrics(y_tr[active_mask], oof_scaffold[active_mask],
                            label="scaffold_bootstrap [active]")
print(f"\nScaffold bootstrap OOF RAE: {m_scaffold['RAE']:.4f}")
print(pd.DataFrame([m_oob, m_scaffold],
                    index=["oob_mean","scaffold_bootstrap"]).round(4).to_string())



=== Scaffold 5-fold CV (bootstrap within each fold) ===


  fold 1  RAE=0.4864  uncertainty_mean=0.1867


  fold 2  RAE=0.5681  uncertainty_mean=0.1847


  fold 3  RAE=0.5910  uncertainty_mean=0.1836


  fold 4  RAE=0.5618  uncertainty_mean=0.1840


  fold 5  RAE=0.5925  uncertainty_mean=0.1830


  [scaffold_bootstrap] RAE=0.5550 MAE=0.5050 R2=0.6088 r=0.7812 rho=0.7384 tau=0.5455
  [scaffold_bootstrap [active]] RAE=3.7839 MAE=0.7935 R2=-9.6192 r=0.0327 rho=0.0931 tau=0.0635

Scaffold bootstrap OOF RAE: 0.5550
                       RAE     MAE      R2  Pearson  Spearman  Kendall
oob_mean            0.5475  0.4981  0.6189   0.7879    0.7476   0.5547
scaffold_bootstrap  0.5550  0.5050  0.6088   0.7812    0.7384   0.5455


In [7]:
# ---- Uncertainty-weighted ensemble: down-weight uncertain predictions ----
# w_i = 1 / (sigma_i + eps); normalize
eps = 0.01  # floor to avoid division by very small sigma
weights_unc = 1.0 / (oof_scaffold_std + eps)
weights_unc = weights_unc / weights_unc.mean()  # normalize

# This doesn't directly give a prediction (we use it for downstream ensemble weighting)
# But we can evaluate: mean vs uncertainty-discounted blend with mean predictor
print("\n=== Uncertainty analysis ===")
print(f"Uncertainty std range: [{oof_scaffold_std.min():.4f}, {oof_scaffold_std.max():.4f}]")
print(f"Mean uncertainty: {oof_scaffold_std.mean():.4f}")
print(f"Correlation (uncertainty vs |error|): "
      f"{np.corrcoef(oof_scaffold_std, np.abs(y_tr - oof_scaffold))[0,1]:.4f}")

# Quantile analysis: is uncertainty calibrated?
for q in [0.25, 0.50, 0.75, 0.90]:
    thresh = np.quantile(oof_scaffold_std, q)
    low_unc = oof_scaffold_std <= thresh
    r_low = rae(y_tr[low_unc], oof_scaffold[low_unc])
    r_high = rae(y_tr[~low_unc], oof_scaffold[~low_unc])
    print(f"  uncertainty <= {q:.0%} quantile ({thresh:.4f}): "
          f"RAE_low={r_low:.4f}  RAE_high={r_high:.4f}")



=== Uncertainty analysis ===
Uncertainty std range: [0.0577, 1.5405]
Mean uncertainty: 0.1844
Correlation (uncertainty vs |error|): 0.3002
  uncertainty <= 25% quantile (0.1346): RAE_low=0.5705  RAE_high=0.5846
  uncertainty <= 50% quantile (0.1732): RAE_low=0.5262  RAE_high=0.6422
  uncertainty <= 75% quantile (0.2212): RAE_low=0.5155  RAE_high=0.7395
  uncertainty <= 90% quantile (0.2666): RAE_low=0.5357  RAE_high=0.7808


In [8]:
# ---- Final test predictions ----
print("\nFinal test predictions from bootstrap ensemble...", flush=True)
te_mean = te_preds_all.mean(0)
te_std  = te_preds_all.std(0)

print(f"Test ensemble stats:")
print(f"  mean: min={te_mean.min():.2f}  med={np.median(te_mean):.2f}  max={te_mean.max():.2f}")
print(f"  std:  min={te_std.min():.4f}  med={np.median(te_std):.4f}  max={te_std.max():.4f}")

te_preds = np.clip(te_mean, y_tr.min()-0.5, y_tr.max()+0.5)

# Use scaffold-CV OOF as the official OOF (more reliable than OOB)
oof = oof_scaffold

np.save(DATA_PROCESSED/"oof_stochastic_ensemble.npy",    oof)
np.save(DATA_PROCESSED/"te_oof_stochastic_ensemble.npy", te_preds)
# Save uncertainty arrays for downstream use
np.save(DATA_PROCESSED/"oof_stochastic_std.npy",  oof_scaffold_std)
np.save(DATA_PROCESSED/"te_oof_stochastic_std.npy", te_std)

sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"102_stochastic_ensemble.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
print(f"\n*** nb102 OOF RAE = {m_scaffold['RAE']:.4f} ***")



Final test predictions from bootstrap ensemble...


Test ensemble stats:
  mean: min=2.47  med=4.91  max=5.78
  std:  min=0.0757  med=0.1889  max=0.6649
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\102_stochastic_ensemble.csv
Test: min=2.47 med=4.91 max=5.78

*** nb102 OOF RAE = 0.5550 ***


In [9]:
# ---- Summary: compare all variants ----
print("\n=== Final Summary ===")
results = pd.DataFrame([
    m_oob,
    m_scaffold,
    m_scaffold_a
], index=["oob_mean","scaffold_bootstrap","scaffold_active"])
print(results.round(4).to_string())
print(f"\nBootstrap ensemble ({N_BOOTSTRAP} models) OOF RAE: {m_scaffold['RAE']:.4f}")
print(f"Uncertainty std (test): mean={te_std.mean():.4f}  med={np.median(te_std):.4f}")



=== Final Summary ===
                       RAE     MAE      R2  Pearson  Spearman  Kendall
oob_mean            0.5475  0.4981  0.6189   0.7879    0.7476   0.5547
scaffold_bootstrap  0.5550  0.5050  0.6088   0.7812    0.7384   0.5455
scaffold_active     3.7839  0.7935 -9.6192   0.0327    0.0931   0.0635

Bootstrap ensemble (50 models) OOF RAE: 0.5550
Uncertainty std (test): mean=0.1978  med=0.1889
